In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
np.random.seed(42)
n = 10000

In [3]:
df = pd.DataFrame({
    'age':np.random.randint(18, 75, n),
    'monthly_spend':np.random.normal(80,24,n),
    'months_active':np.random.randint(1,121,n),
    'support_calls':np.random.poisson(2,n),
    'late_payments':np.random.poisson(1,n),
    'satisfaction':np.random.randint(1,11,n),
     'contract_type': np.random.choice(
        ['Monthly', 'Yearly', 'Two-Year'],
        n,
        p=[0.55, 0.30, 0.15]
    ),
    'internet_type': np.random.choice(
        ['Fiber', 'DSL', 'None'],
        n,
        p=[0.55, 0.35, 0.10]
    ),
    'discount': np.random.randint(0, 31, n)
})


In [4]:
df

,age,monthly_spend,months_active,support_calls,late_payments,satisfaction,contract_type,internet_type,discount
0,56,56.289617,82,6,2,6,Yearly,Fiber,12
1,69,81.544942,30,2,1,6,Two-Year,DSL,25
2,46,37.269028,72,1,0,8,Monthly,None,10
3,32,88.184924,105,2,0,5,Monthly,Fiber,5
4,60,40.849412,115,3,0,5,Yearly,None,0
...,...,...,...,...,...,...,...,...,...
9995,40,67.581400,111,2,1,6,Yearly,DSL,1
9996,58,50.575138,72,2,0,1,Monthly,Fiber,9
9997,57,66.092425,54,0,0,10,Two-Year,Fiber,6
9998,46,129.605792,9,1,1,3,Monthly,Fiber,9


In [5]:
risk = (
    (10 - df['satisfaction']) * 0.35 +
    df['support_calls'] * 0.25 +
    df['late_payments'] * 0.4 +
    (df['contract_type'] == 'Monthly') * 0.8 -
    df['months_active'] * 0.01
)

In [6]:
probability = 1 / (1 + np.exp(-(risk - 3)))

In [7]:
df['churn'] = np.random.binomial(1, probability)

In [8]:
df

,age,monthly_spend,months_active,support_calls,late_payments,satisfaction,contract_type,internet_type,discount,churn
0,56,56.289617,82,6,2,6,Yearly,Fiber,12,0
1,69,81.544942,30,2,1,6,Two-Year,DSL,25,0
2,46,37.269028,72,1,0,8,Monthly,None,10,0
3,32,88.184924,105,2,0,5,Monthly,Fiber,5,0
4,60,40.849412,115,3,0,5,Yearly,None,0,0
...,...,...,...,...,...,...,...,...,...,...
9995,40,67.581400,111,2,1,6,Yearly,DSL,1,0
9996,58,50.575138,72,2,0,1,Monthly,Fiber,9,1
9997,57,66.092425,54,0,0,10,Two-Year,Fiber,6,0
9998,46,129.605792,9,1,1,3,Monthly,Fiber,9,1


In [10]:
df['churn'].value_counts()

churn
0    6283
1    3717
Name: count, dtype: int64

In [11]:
df['churn'].value_counts(normalize=True) * 100

churn
0    62.83
1    37.17
Name: proportion, dtype: float64

In [17]:
cat_cols = ['contract_type', 'internet_type']

In [16]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')

In [18]:
X_train_encoded = ohe.fit_transform(X_train[cat_cols])
X_test_encoded = ohe.transform(X_test[cat_cols])
X_train = X_train.drop(columns=cat_cols)
X_test = X_test.drop(columns=cat_cols)
X_train = X_train.join(X_train_encoded)
X_test = X_test.join(X_test_encoded)

In [21]:
X_train

,age,monthly_spend,months_active,support_calls,late_payments,satisfaction,discount,contract_type_Monthly,contract_type_Two-Year,contract_type_Yearly,internet_type_DSL,internet_type_Fiber,internet_type_None
9254,59,60.200900,61,1,1,2,19,1.0,0.0,0.0,1.0,0.0,0.0
1561,73,87.191646,97,2,2,3,23,1.0,0.0,0.0,0.0,1.0,0.0
1670,62,111.604536,108,1,1,9,13,1.0,0.0,0.0,1.0,0.0,0.0
6087,51,70.721216,49,1,0,8,19,0.0,1.0,0.0,0.0,1.0,0.0
6669,23,79.473168,31,2,3,9,9,0.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,28,102.679738,11,2,1,10,20,0.0,0.0,1.0,0.0,1.0,0.0
5191,68,76.941986,96,1,0,2,25,1.0,0.0,0.0,0.0,1.0,0.0
5390,74,80.779280,70,2,1,7,14,0.0,0.0,1.0,1.0,0.0,0.0
860,45,88.863115,81,1,4,10,17,1.0,0.0,0.0,0.0,1.0,0.0


In [12]:
X = df.drop('churn', axis=1)
y = df['churn']

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
model = XGBClassifier()

In [22]:
pipe = Pipeline([('model', model)])

In [23]:
param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}

In [24]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=3
)

In [25]:
grid.fit(X_train, y_train)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV 1/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=0.8;, score=0.768 total time=   0.3s
[CV 2/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=0.8;, score=0.773 total time=   0.2s
[CV 3/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=0.8;, score=0.781 total time=   0.2s
[CV 1/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=1.0;, score=0.768 total time=   0.2s
[CV 2/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=1.0;, score=0.771 total time=   0.2s
[CV 3/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=200, model__subsample=1.0;, score=0.781 total time=   0.2s
[CV 1/3] END model__learning_rate=0.05, model__max_depth=3, model__n_estimators=400, model__subsample=0.8

,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [3, 5], 'model__n_estimators': [200, 400], 'model__subsample': [0.8, 1.0]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [26]:
y_pred = grid.predict(X_test)

In [27]:
print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

Best params: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best CV accuracy: 0.7738197361768545


In [28]:
from sklearn.metrics import classification_report,accuracy_score

In [29]:
print('Test accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))

Test accuracy: 0.73
              precision    recall  f1-score   support

           0       0.75      0.85      0.80      1243
           1       0.68      0.53      0.60       757

    accuracy                           0.73      2000
   macro avg       0.72      0.69      0.70      2000
weighted avg       0.72      0.73      0.72      2000



In [30]:
from sklearn.metrics import roc_auc_score

y_prob = grid.best_estimator_.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)

ROC-AUC: 0.7794258149467931


In [31]:
best_model = grid.best_estimator_

In [32]:
import joblib

In [33]:
joblib.dump(best_model, 'clientMLprediction.pkl')

['clientMLprediction.pkl']

In [34]:
joblib.dump(ohe, 'churn_ohe.pkl')

['churn_ohe.pkl']